In [ ]:
import pandas as pd
import numpy as np
import re
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.en.stop_words import STOP_WORDS

df = pd.read_csv("../data/data_preprocessed_final.csv", sep=",")
df.sample(5)

,index,avis,dates,series,allocine_id,review_id,plateforme,notes,notes_3,nb_capital_words,nb_exclamation_points,nb_exclamation_points_chains,nb_interrogation_points,nb_interrogation_points_chains,lenght,tokens,tokens_lemmatized,sentiment
480,481,A lotus in the middle of a swamp - It's good t...,2026-02-26,A Knight of the Seven Kingdoms,28230,35604692,Imdb,10,1,1,0,0,0,0,1253,lotus middle swamp good see franchise return r...,lotus middle swamp good see franchise return r...,21
137,137,Garbage - This is the most ridiculous shows I ...,2026-01-20,A Knight of the Seven Kingdoms,28230,35604348,Imdb,1,-1,1,0,0,0,0,316,garbage ridiculous shows seen long time relies...,garbage ridiculous show seen long time relies ...,3
75,75,"Fantastic start, with some minor annoyances s0...",2026-01-20,A Knight of the Seven Kingdoms,28230,35604286,Imdb,10,1,0,0,0,0,0,53,fantastic start minor annoyances s01e01,fantastic start minor annoyance s01e01,2
158,158,worst interpretation of the GoT universe - I h...,2026-01-26,A Knight of the Seven Kingdoms,28230,35604369,Imdb,1,-1,0,0,0,0,0,350,worst interpretation got universe hope eventua...,worst interpretation got universe hope eventua...,-2
1403,1405,Too far from history - Could have done a bette...,2026-01-15,King & Conqueror,35604,35605616,Imdb,5,0,0,0,0,0,0,353,far history could done better job aligned reco...,far history could done better job aligned reco...,0


In [94]:
# A faire manuellement pour tous chaque série souhaitée
print(df.info(), "\n")

display(df.sample(5))

<class 'pandas.DataFrame'>
RangeIndex: 1410 entries, 0 to 1409
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   index                           1410 non-null   int64
 1   avis                            1410 non-null   str  
 2   dates                           1410 non-null   str  
 3   series                          1410 non-null   str  
 4   allocine_id                     1410 non-null   int64
 5   review_id                       1410 non-null   int64
 6   plateforme                      1410 non-null   str  
 7   notes                           1410 non-null   int64
 8   notes_3                         1410 non-null   int64
 9   nb_capital_words                1410 non-null   int64
 10  nb_exclamation_points           1410 non-null   int64
 11  nb_exclamation_points_chains    1410 non-null   int64
 12  nb_interrogation_points         1410 non-null   int64
 13  nb_interrogati

,index,avis,dates,series,allocine_id,review_id,plateforme,notes,notes_3,nb_capital_words,nb_exclamation_points,nb_exclamation_points_chains,nb_interrogation_points,nb_interrogation_points_chains,lenght,tokens,tokens_lemmatized,sentiment
1199,1201,The best one yet from GOT universe. - Loved th...,2026-02-21,A Knight of the Seven Kingdoms,28230,35605412,Imdb,8,1,2,1,0,0,0,414,best one yet got universe loved show exception...,best one yet got universe loved show exception...,21
850,852,"Perfection!! - When they announced this show, ...",2026-02-23,A Knight of the Seven Kingdoms,28230,35605063,Imdb,10,1,0,2,1,0,0,421,perfection announced show expect much alas epi...,perfection announced show expect much ala epis...,10
706,707,Greatness - This episode was absolutely phenom...,2026-02-22,A Knight of the Seven Kingdoms,28230,35604918,Imdb,10,1,0,0,0,0,0,866,greatness episode absolutely phenomenal unlike...,greatness episode absolutely phenomenal unlike...,18
284,284,"So good - As often with big franchise, this is...",2026-02-07,A Knight of the Seven Kingdoms,28230,35604495,Imdb,8,1,0,1,0,0,0,330,good often big franchise side stories shine le...,good often big franchise side story shine less...,20
1149,1151,...but an highly unintelligent script -,2026-02-24,A Knight of the Seven Kingdoms,28230,35605362,Imdb,7,1,0,0,0,0,0,40,highly unintelligent script,highly unintelligent script,-2


In [95]:
# Modélisation

# Isoler les verbatims dans une variable    
docs = df["avis"]

# Prétraitement simple
def preprocess(texts):
    cleaned = []
    for t in texts:
        t = t.lower()                       # Mettre en minuscules
        t = re.sub(r"\s+", " ", t).strip()  # Normaliser les espaces
        if t:
            cleaned.append(t)
    return cleaned

verbatims = preprocess(docs)

# Modèle de traitement des verbatims
embedding_model = SentenceTransformer("all-mpnet-base-v2")

embeddings = embedding_model.encode(verbatims, show_progress_bar=True)

# Doc : https://maartengr.github.io/BERTopic/#installation
# Doc : https://www.metricsmag.com/comment-bertopic-ameliore-le-topic-modeling/
# Doc : https://medium.com/@tiffanyccchen/practical-tips-for-better-topic-modeling-using-bertopic-d12daf347918
# Doc : https://www.begenai.com/comment-bertopic-produit-il-des-topics-interpretables/


Batches: 100%|██████████| 45/45 [05:21<00:00,  7.15s/it]


In [96]:

# Fixer les conditions UMAP, notamment pour permettre la reproductibilité
umap_model = UMAP(
    n_neighbors=5,               # Monter à 50 maximum s'il y a trop de clusters
    n_components=5, 
    min_dist=0.0, 
    metric='cosine', 
    random_state=42               # Permet la reproductibilité
)

# Configurer HDBSCAN avec min_cluster_size pour réduire le nombre de topics
hdbscan_model = HDBSCAN(
    min_cluster_size=15,                # Nombre minimum de verbatims par topic
    min_samples=5,                      # Optionnel : contrôle la sensibilité au bruit
    cluster_selection_method="eom",    # cluster plus fins et homogènes
    prediction_data=True
)

# Améliorer la représentation des topics
vectorizer_model = CountVectorizer(stop_words=list(STOP_WORDS), ngram_range=(1, 2))
representation_model = KeyBERTInspired()

# Entrainement du modèle BERTopic
topic_model = BERTopic(embedding_model=embedding_model,
                       umap_model=umap_model, 
                       hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer_model,
                       representation_model=representation_model)

topics, probs = topic_model.fit_transform(verbatims, embeddings=embeddings)

# Doc : https://maartengr.github.io/BERTopic/#installation

In [97]:
# Récupération des mots-clés par topic
topic_info = topic_model.get_topic_info()
print(topic_info)

    Topic  Count                                               Name  \
0      -1    249    -1_game thrones_thrones_westeros_seven kingdoms   
1       0    162            0_thrones_game thrones_episodes_episode   
2       1    150       1_episode_characters_historical drama_dramas   
3       2    110            2_episode_game thrones_thrones_episodes   
4       3     98            3_game thrones_thrones_episodes_episode   
5       4     52           4_episode_episodes_genuinely good_season   
6       5     49  5_episodes series_episode_episodes_episodes short   
7       6     48  6_game thrones_thrones_absolute cinema_cinema ...   
8       7     43        7_episode_episodes season_episodes_like got   
9       8     40       8_game thrones_thrones_thrones house_episode   
10      9     39     9_episode_episodes_episodes short_episode dunk   
11     10     36           10_game thrones_thrones_episode_westeros   
12     11     36       11_seasons got_episode_got series_unlike got   
13    

In [9]:
# Cliquer ici si besoin de réinitialiser le nombre d'outilers pour itérer
topic_model.update_topics(verbatims, topics=topics)

2026-11-11 11:58:06,495 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [98]:
# Réduction du nombre d'outliers
print("Nb outlier initialement : ", np.sum(np.array(topic_model.topics_) == -1), "\n")

nouveaux_topics = topic_model.reduce_outliers(
    verbatims,
    topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.5
)

topic_model.update_topics(verbatims, topics=nouveaux_topics)

nouveaux_topics_info = topic_model.get_topic_info()

print("Nb outliers après : ", np.sum(np.array(topic_model.topics_) == -1), "\n")
print(topic_model.get_topic_info())

2026-11-14 17:28:32,598 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Nb outlier initialement :  249 

Nb outliers après :  30 

    Topic  Count                                   Name  \
0      -1     30     -1_cinema_functions_bodily_writing   
1       0    176                        0_the_of_and_it   
2       1    152                    1_history_the_to_it   
3       2    148                        2_the_it_of_and   
4       3    101                    3_the_of_and_knight   
5       4     59                      4_and_the_is_this   
6       5     69                    5_the_to_it_episode   
7       6     54                       6_the_and_you_is   
8       7     60                      7_got_to_this_the   
9       8     48                       8_the_of_it_this   
10      9     46                     9_egg_and_the_dunk   
11     10     44                       10_the_of_to_and   
12     11     56                    11_got_this_the_and   
13     12     44                    12_dunk_and_the_egg   
14     13     41                    13_the_got_this_and 

In [99]:
# Visualisation des verbatims par topic
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)

topic_model.visualize_documents(verbatims, reduced_embeddings=reduced_embeddings)

# Le graphique est interactif : il est possible de zoomer pour voir une zone de plus près.

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'hovertext': array(['poor writing from martin -', 'absolute cinema -', 'why 9? -',
                                  "soap opera video quality so i didn't even watch! - i'm so happy to see the poor reviews about the writing and such, so i didn't have to spend any more time than necessary looking at it. i only made it a few minutes into the movie, because i refuse to watch anything that looks like production had to borrow the daytime television cameras. the cinematography, video, and overall visual presentation is terrible. i can't be the only person who feels this way!?!?",
                                  'a smaller story that still feels epic -',
                                  'an attempt at comedy drama that comes off as a parody. -',
                                  'intimate story big screen energy -', 'this is cinema -',
                                  'is this saturday live sketch? -', 'absolute cinema -', 'grey area -',
                                  'dunk+egg=bff -', 'absolute cinema -',
                                  'so faithful to the source material -',
                                  'good, but unnecessary bodily functions -',
                                  'big heart, sharp wit - but give us the full hour -', 'best of best -',
                                  'unnecessary bodily functions -', 'hype for what?? -',
                                  'authors make or break a story - as we heard r. r is personally involved with the story and immediately we can feel the presence of a writer/creator handling the writing and story building smoothly and ever more profound as the episodes unfold. great writing will make direction easier to steer and acting more level with the world itself. i liken this to eichiro oda personally involved in the live action one piece of his own animated series/manga adaption. when great minds are involved, especially when the great mind is the one who is the creator, the comparison to stories that are not handled with such authority seem to show a different level of quality and substance. we should be more focused on who is writing a story and who said writer wants to direct the vision and thus the acting will provide stability accordingly.',
                                  'was it really necessary to poop on tv ? -', 'sounds great -',
                                  'a great story that recovers from a rough start -',
                                  'nothing happens! -', 'small scale, big heart -', 'absolute cinema -',
                                  'defecating and urinating is not entertainment -',
                                  'dunk & egg dynamic -',
                                  'i never review things, but this is just that good. -',
                                  'leave the bodily functions out! -', nan], dtype=object),
              'marker': {'color': '#CFD8DC', 'opacity': 0.5, 'size': 5},
              'mode': 'markers+text',
              'name': 'other',
              'showlegend': False,
              'type': 'scattergl',
              'x': {'bdata': ('oOPyP59OKUA1GX8+ZnDhvu5OB0CkmM' ... 'Di27M/X9UAwM/RpjzHULc/vICzPw=='),
                    'dtype': 'f4'},
              'y': {'bdata': ('+ZyqQGuBUUBpiX1AXMyZQANBTUCrXa' ... 'ACjY1AsdMWv2Xwi0Bef45ACUZeQA=='),
                    'dtype': 'f4'}},
             {'hoverinfo': 'text',
              'hovertext': array(["a spot-on gem for westeros fans - as someone who hasn't read george r. r. martin's tales of dunk and egg novellas, the source for this series, a knight of the seven kingdoms blew me away. hbo nailed the adaptation, staying book-accurate (from what i've gathered) with stunning attention to detail in costumes, sets, and that gritty westeros vibe. peter claffey as dunk and dexter sol ansell as egg deliver magnetic chemistry, their hedge knight and squire duo feels authentic, flawed, and full of heart, blending humor, chivalry, and

In [100]:
# Création du dataframe
df_topics = pd.DataFrame({
    "text" : verbatims,
    "topic" : nouveaux_topics,
    "probability" : probs,
    "review_id" : df["review_id"]
})

# Affichage les résultats
print(df_topics)

                                                   text  topic  probability  \
0     honest opinion - turns out westeros is way mor...      8     0.921000   
1     so far, so good (don't screw it up) - the firs...      3     1.000000   
2     a proper return to westeros - i stayed up for ...     10     1.000000   
3     the only disappointment i had with ep 1 - its ...     10     1.000000   
4     really freaking good with only one bad moment ...      2     0.934935   
...                                                 ...    ...          ...   
1405  mediocre filmmaking - weak script in combinati...      1     1.000000   
1406  drunk history - as many other reviewers have n...      1     1.000000   
1407                                      i get it... -     15     0.862415   
1408  realism... - earl's military crews count 5-8 s...      1     1.000000   
1409  historical inaccuracies - the historical inacc...      1     1.000000   

      review_id  
0      35604211  
1      35604212

In [101]:
# Vue d'ensemble des non classés
df_topics[df_topics["topic"] == -1]

,text,topic,probability,review_id
35,absolute cinema -,-1,0.0,35604246
94,authors make or break a story - as we heard r....,-1,0.0,35604305
127,leave the bodily functions out! -,-1,0.0,35604338
190,"big heart, sharp wit - but give us the full ho...",-1,0.0,35604401
205,a smaller story that still feels epic -,-1,0.0,35604416
276,was it really necessary to poop on tv ? -,-1,0.0,35604487
343,"small scale, big heart -",-1,0.0,35604555
391,dunk+egg=bff -,-1,0.0,35604603
410,intimate story big screen energy -,-1,0.0,35604622
429,"good, but unnecessary bodily functions -",-1,0.0,35604641


In [ ]:
# Export csv
df_topics.to_csv("../data/topics.csv", sep=";", index=False)
nouveaux_topics_info.to_csv("../data/topics_names.csv", sep=";", encoding="utf-8-sig", index=False)


Piste abandonnée car pas intéressante : difficile d'isoler les topics, surtout pour A Knight of the Seven Kingdoms, car la série a été très bien accueillie et les commentaires se ressemblent globalement.

Pour King & Conqueror, les topics font ressostir le manque de précision historique et une faiblesse au niveau de l'écriture.